# Tarea 1: Predicción de resultados del fútbol uruguayo

In [ ]:
# %pip install pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

--- 
### 1. Carga del Dataset y Descripción de Atributos

In [22]:
DATASET_FILE = "./futbol_uruguayo.csv" 

dataset = pd.read_csv(DATASET_FILE)
dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


#### Descripción de los atributos:

| Atributo | Descripción |
| :--- | :--- |
| **`home`** | Nombre del equipo local (no necesariamente único) |
| **`away`** | Nombre del equipo visitante (no necesariamente único) |
| **`date`** | Fecha del partido |
| **`gh`** | Goles del equipo local (incluyendo tiempo extra y penales) |
| **`ga`** | Goles del equipo visitante (incluyendo tiempo extra y penales) |
| **`full_time`** | "F"=el partido terminó en 90', "E"=tiempo extra, "P"=penales |
| **`competition`** | Nombre del país de la liga o nombre de la competición int. |
| **`home_ident`** | Identificador único del equipo local |
| **`away_ident`** | Identificador único del equipo visitante |
| **`home_country`** | País del equipo local |
| **`away_country`** | País del equipo visitante |
| **`home_code`** | Código de país del equipo local |
| **`away_code`** | Código de país del equipo visitante |
| **`home_continent`** | Continente del equipo local |
| **`away_continent`** | Continente del equipo visitante |
| **`continent`** | Continente de la competición |
| **`level`** | "national"= liga local, "international"= copa internacional |

--- 
### 2. Definición de la Variable Objetivo (`ganador`)

El objetivo del modelo es predecir el resultado final de un partido de fútbol, clasificándolo en una de tres categorías posibles: victoria local, victoria visitante o empate.

Dado que el dataset original no incluye directamente una columna de resultado, se deduce la variable objetivo **`ganador`** mediante la comparación de los goles anotados por el equipo local (`gh`) y el visitante (`ga`):

* Si $\text{gh} > \text{ga} \implies$ **`L`**
* Si $\text{ga} > \text{gh} \implies$ **`V`**
* Si $\text{gh} == \text{ga} \implies$ **`E`**

Concretado lo anterior, la información que proveen los atributos "gh" y "ga" ya se ve contemplada por la variable objetivo. Por lo tanto, su presencia en el dataset no agrega significancia al entrenamiento del modelo y se deben remover ambos atributos del dataset.

In [23]:
dataset["ganador"] = np.select(
    [
        dataset["gh"] > dataset["ga"],
        dataset["gh"] < dataset["ga"],
        dataset["gh"] == dataset["ga"],
    ],
    [
        "L",
        "V",
        "E",
    ],
    default="sin_dato",
)

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level,ganador
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,V
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,E
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L


--- 
### 3. Selección de Atributos

En esta etapa del preprocesamiento se realiza una selección de los atributos del dataset con el objetivo de maximizar la capacidad de aprendizaje del modelo en la clasificación de las instancias. Para ello, se examina cada atributo de forma individual para evaluar si contribuye de manera significativa al entrenamiento o si, dadas sus características, puede ser removido sin perjudicar el desempeño.

* **Atributos constantes:** No contribuyen al aprendizaje del modelo debido a que mantienen el mismo valor para todas las instancias del dataset (varianza cero).
* **Atributos redundantes:** Representan valores equivalentes dentro del dataset, por lo que basta con conservar uno de ellos.

#### A. Detección de Atributos Constantes

In [4]:
# Detección de Atributos Constantes
constantes = dataset.nunique(dropna=False)[dataset.nunique(dropna=False) == 1]
print("Atributos constantes detectados:")
print(constantes)

Atributos constantes detectados:
competition       1
home_country      1
away_country      1
home_code         1
away_code         1
home_continent    1
away_continent    1
continent         1
level             1
dtype: int64


Los atributos constantes son los siguientes:
* `competition`, `home_country`, `away_country` (Todos refieren a Uruguay).
* `home_code`, `away_code` (Código constante `UY`).
* `home_continent`, `away_continent`, `continent` (Todos refieren a `South America`).
* `level` (Constante con el valor `national`).

Estos atributos no serán incluidos en los conjuntos de entrenamiento y evaluación.

#### B. Detección de Atributos Redundantes

In [5]:
# Verificar cuántos identificadores tiene cada nombre de equipo, y viceversa
home_name_to_id = dataset.groupby("home")["home_ident"].nunique(dropna=False)
home_id_to_name = dataset.groupby("home_ident")["home"].nunique(dropna=False)

away_name_to_id = dataset.groupby("away")["away_ident"].nunique(dropna=False)
away_id_to_name = dataset.groupby("away_ident")["away"].nunique(dropna=False)

print(f"Cada valor de home se corresponde a un solo valor de home_ident, y viceversa: ", (home_name_to_id == 1).all()  & (home_id_to_name == 1).all())
print(f"Cada valor de away se corresponde a un solo valor de away_ident, y viceversa: ", (away_name_to_id == 1).all()  & (away_id_to_name == 1).all())

Cada valor de home se corresponde a un solo valor de home_ident, y viceversa:  True
Cada valor de away se corresponde a un solo valor de away_ident, y viceversa:  True


Existe una correspondencia 1:1, mantener ambas variables introduciría información redundante. Elegimos quedarnos con `home` y `away` y se descartan sus identificadores (`home_ident` y `away_ident`).

#### C. Descomposición del atributo `date`

El atributo `date` se divide en los atributos `day`, `month` y `year` para representar sus componentes por separado. Esta transformación facilita que el modelo identifique patrones relacionados con el momento del año en que se disputó el partido, como diferencias entre meses, temporadas o períodos históricos. Además, evita tratar cada fecha completa como un valor independiente, lo que podría dificultar el aprendizaje. Una vez extraídos estos componentes, se debe remover el atributo `date` original para evitar mantener información redundante.

En el flujo final, este procedimiento será ejecutado dentro del `Pipeline`.

In [24]:
dataset["date"] = pd.to_datetime(dataset["date"])
dataset["day"] = dataset["date"].dt.day
dataset["month"] = dataset["date"].dt.month
dataset["year"] = dataset["date"].dt.year

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,...,home_code,away_code,home_continent,away_continent,continent,level,ganador,day,month,year
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,V,5,3,1932
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,E,5,3,1932
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932


---
### 4. Análisis de Balance y Estratificación:

In [7]:
print(f"Cantidad total de instancias: {dataset.shape[0]}")
print(f"Cantidad total de atributos: {dataset.shape[1]}")

Cantidad total de instancias: 15207
Cantidad total de atributos: 21


In [8]:
# Cálculo de la dispersión de instancias respecto a las clases
dataset["ganador"].value_counts(normalize=True).mul(100).round(2)

ganador
L    44.35
V    28.20
E    27.44
Name: proportion, dtype: float64

La distribución de la clase objetivo muestra las siguientes proporciones:
* **`L`**: **44.35%**
* **`V`**: **28.20%**
* **`E`**: **27.44%**

Aunque existe un predominio de las victorias locales, la distribución no presenta un desbalance crítico (como ocurriría en escenarios con clases minoritarias $< 5\%$), por lo que no se requiere la aplicación de técnicas de mitigación como *SMOTE* o *undersampling*. 

Sin embargo, para evitar que una división puramente aleatoria, se aplica un muestreo estratificado (`stratify=dataset_Y`). De esta manera, se garantiza que tanto el conjunto de entrenamiento como el de evaluación mantengan proporciónes similares a las originales del dataset para cada clase.

---
### 5. División del Conjunto de Datos (`train_test_split`)

Se debe dividir el conjunto dataset de la siguiente forma:

Conjunto de entrenamiento los partidos jugados hasta el año 2023 inclusive y como conjunto de evaluación los partidos jugados en 2024 y 2025.


In [25]:
# Instancias con partidos jugados hasta el año 2023 inclusive
df_entrenamiento = dataset[dataset['year'] <= 2023]
X_train = df_entrenamiento.drop(columns=["ganador", "gh", "ga"])
Y_train = df_entrenamiento["ganador"]

df_evaluacion = dataset[dataset['year'] > 2023]
X_test = df_evaluacion.drop(columns=["ganador", "gh", "ga"])
Y_test = df_evaluacion["ganador"]

print(f"Dimensiones de X_train (Entrenamiento): {X_train.shape}")
print(f"Dimensiones de X_test (Evaluación):    {X_test.shape}")

Dimensiones de X_train (Entrenamiento): (14734, 18)
Dimensiones de X_test (Evaluación):    (473, 18)


In [26]:
numeric_features = ['year', 'month', 'day']
categorical_features = ['home', 'away', 'full_time']

# Pipeline para los atributos numéricos
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer()) # Reemplaza los valores NaN por la media (mean) por defecto
])

# Pipeline para los atributos categóricos
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(fill_value='unknown',strategy='constant')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop' # Elimina automáticamente todas las demás columnas no especificadas
)


In [11]:
MIN_INFO_GAIN = 0.005

In [12]:

pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DecisionTreeClassifier(
        criterion='entropy',
        min_impurity_decrease=MIN_INFO_GAIN,
        random_state=62
    ))
])

Naive Bayes.

En este seccion se realizará el entrenamiento en busqueda de los mejores hiperparametros para el algoritmo.

In [27]:
class NaiveBayes:
    """Naive Bayes categórico multiclase con suavizado m-estimate."""

    def __init__(self, m=1.0):
        self.m = m

    def fit(self, X, y):
        self.clases_ = sorted(y.unique())
        cantidad_total = len(y)

        self.log_prioris_ = {}
        self.tablas_categoricas_ = {}    # tablas_categoricas_[clase][columna][valor] = probabilidad
        self.respaldo_categoricas_ = {}  # probabilidad para valores no vistos en entrenamiento
        self.dominio_columnas_ = {columna: X[columna].unique().tolist() for columna in X.columns}

        for clase in self.clases_:
            mascara_clase = (y == clase)
            X_clase = X.loc[mascara_clase]
            cantidad_clase = mascara_clase.sum()
            self.log_prioris_[clase] = np.log(cantidad_clase / cantidad_total)

            self.tablas_categoricas_[clase] = {}
            self.respaldo_categoricas_[clase] = {}
            for columna in X.columns:
                dominio = self.dominio_columnas_[columna]
                probabilidad_uniforme = 1.0 / len(dominio)
                conteos = X_clase[columna].value_counts()

                tabla_columna = {
                    valor: (conteos.get(valor, 0) + self.m * probabilidad_uniforme) / (cantidad_clase + self.m)
                    for valor in dominio
                }
                self.tablas_categoricas_[clase][columna] = tabla_columna
                # Probabilidad para un valor jamás visto en entrenamiento (conteo = 0)
                self.respaldo_categoricas_[clase][columna] = (self.m * probabilidad_uniforme) / (cantidad_clase + self.m)

        return self

    def _log_verosimilitud(self, fila, clase):
        log_probabilidad = 0.0
        for columna, tabla_columna in self.tablas_categoricas_[clase].items():
            valor = fila[columna]
            log_probabilidad += np.log(tabla_columna.get(valor, self.respaldo_categoricas_[clase][columna]))
        return log_probabilidad

    def predict_log_proba(self, X):
        filas = [
            {
                clase: self.log_prioris_[clase] + self._log_verosimilitud(fila, clase)
                for clase in self.clases_
            }
            for _, fila in X.iterrows()
        ]
        return pd.DataFrame(filas, index=X.index)[self.clases_]

    def predict(self, X):
        log_probabilidades = self.predict_log_proba(X)
        return log_probabilidades.idxmax(axis=1)


In [28]:
# Solo se usan atributos categóricos (día, mes y año no aportan señal para el resultado)
atributos_naive_bayes = categorical_features

X_train_nb = X_train[atributos_naive_bayes].fillna('desconocido')
X_test_nb = X_test[atributos_naive_bayes].fillna('desconocido')



X_train_nb.head()


,home,away,full_time
0,Bella Vista,Defensor Sporting,F
1,CA Penarol,River Plate,F
2,Montevideo Wanderers,Racing Club,F
3,Central Espanol,Rampla Juniors Futbol Club,F
4,Nacional,Institucion Atletica Sud America,F


In [29]:
naive_bayes = NaiveBayes(m=1.0)
naive_bayes.fit(X_train_nb, Y_train)

predicciones_nb = naive_bayes.predict(X_test_nb)
predicciones_nb.head()


14734    L
14735    V
14736    L
14737    V
14738    L
dtype: object

In [30]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(Y_test, predicciones_nb))
print()
print(classification_report(Y_test, predicciones_nb))
print("Matriz de confusión (orden de clases:", naive_bayes.clases_, "):")
print(confusion_matrix(Y_test, predicciones_nb, labels=naive_bayes.clases_))


Accuracy: 0.4820295983086681

              precision    recall  f1-score   support

           E       0.42      0.04      0.07       132
           L       0.46      0.85      0.60       190
           V       0.56      0.41      0.48       151

    accuracy                           0.48       473
   macro avg       0.48      0.43      0.38       473
weighted avg       0.48      0.48      0.41       473

Matriz de confusión (orden de clases: ['E', 'L', 'V'] ):
[[  5 104  23]
 [  4 161  25]
 [  3  86  62]]


---
### 6. Selección de Hiperparámetros con Cross-Validation

Se aplica **validación cruzada temporal** sobre el conjunto de **entrenamiento** (nunca sobre el conjunto de evaluación 2024-2025, para no filtrar información) con el objetivo de elegir el mejor valor del hiperparámetro `m` (suavizado m-estimate) del `NaiveBayes` propio.

* Valores de `m` evaluados: `[0.1, 0.5, 1, 2, 5, 10, 20, 50]`.
* Se usa `TimeSeriesSplit` con 5 particiones, respetando el orden temporal de los partidos.
* En cada partición se entrena con partidos anteriores y se valida con partidos posteriores.
* Métrica de selección: **F1-macro promedio** entre las particiones (pondera por igual a las 3 clases, sin favorecer a `L` por ser mayoritaria). También se registra el accuracy como referencia.


In [38]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score

valores_m = [0.1, 0.5, 1, 2, 5, 10, 20, 50]
cantidad_folds = 5

kf = TimeSeriesSplit(n_splits=cantidad_folds)

resultados_cv = []
for m in valores_m:
    f1_folds = []
    accuracy_folds = []
    for indice_train, indice_val in kf.split(X_train_nb):
        X_train_fold = X_train_nb.iloc[indice_train]
        X_val_fold = X_train_nb.iloc[indice_val]
        y_train_fold = Y_train.iloc[indice_train]
        y_val_fold = Y_train.iloc[indice_val]

        modelo_fold = NaiveBayes(m=m)
        modelo_fold.fit(X_train_fold, y_train_fold)
        predicciones_fold = modelo_fold.predict(X_val_fold)

        f1_folds.append(f1_score(y_val_fold, predicciones_fold, average='macro'))
        accuracy_folds.append(accuracy_score(y_val_fold, predicciones_fold))

    resultados_cv.append({
        'm': m,
        'f1_macro_promedio': np.mean(f1_folds),
        'f1_macro_std': np.std(f1_folds),
        'accuracy_promedio': np.mean(accuracy_folds),
    })

resultados_cv_df = pd.DataFrame(resultados_cv).sort_values('f1_macro_promedio', ascending=False).reset_index(drop=True)
resultados_cv_df


,m,f1_macro_promedio,f1_macro_std,accuracy_promedio
0,2.0,0.371330,0.023350,0.455560
1,1.0,0.371326,0.023349,0.455560
2,10.0,0.371194,0.023331,0.455642
3,0.5,0.371155,0.023271,0.455316
4,0.1,0.371155,0.023271,0.455316
5,5.0,0.370915,0.023169,0.455316
6,20.0,0.370909,0.023996,0.456456
7,50.0,0.370573,0.025125,0.456782


In [39]:
mejor_fila = resultados_cv_df.iloc[0]
mejor_m = mejor_fila['m']

print(f"Mejor hiperparámetro encontrado: m = {mejor_m}")
print(f"F1-macro promedio (CV): {mejor_fila['f1_macro_promedio']:.4f}")
print(f"Accuracy promedio (CV): {mejor_fila['accuracy_promedio']:.4f}")


Mejor hiperparámetro encontrado: m = 2.0
F1-macro promedio (CV): 0.3713
Accuracy promedio (CV): 0.4556


#### Entrenamiento final con el mejor `m` encontrado

Se reentrena el `NaiveBayes` propio con **todo** el conjunto de entrenamiento (sin folds) usando `mejor_m`, y se evalúa sobre el conjunto de evaluación real (2024-2025), para comparar contra el modelo baseline (`m=1.0`).


In [33]:
naive_bayes_optimizado = NaiveBayes(m=mejor_m)
naive_bayes_optimizado.fit(X_train_nb, Y_train)

predicciones_nb_optimizado = naive_bayes_optimizado.predict(X_test_nb)

print(f"Accuracy (m={mejor_m}):", accuracy_score(Y_test, predicciones_nb_optimizado))
print()
print(classification_report(Y_test, predicciones_nb_optimizado))
print("Matriz de confusión (orden de clases:", naive_bayes_optimizado.clases_, "):")
print(confusion_matrix(Y_test, predicciones_nb_optimizado, labels=naive_bayes_optimizado.clases_))


Accuracy (m=0.1): 0.4820295983086681

              precision    recall  f1-score   support

           E       0.42      0.04      0.07       132
           L       0.46      0.85      0.60       190
           V       0.56      0.41      0.48       151

    accuracy                           0.48       473
   macro avg       0.48      0.43      0.38       473
weighted avg       0.48      0.48      0.41       473

Matriz de confusión (orden de clases: ['E', 'L', 'V'] ):
[[  5 104  23]
 [  4 161  25]
 [  3  86  62]]


#### Cómo leer la matriz de confusión en este problema

`confusion_matrix(Y_test, predicciones, labels=['E', 'L', 'V'])` arma una matriz de 3x3 donde:

* **Las filas representan la clase real** (lo que efectivamente pasó en el partido), en el orden `['E', 'L', 'V']`.
* **Las columnas representan la clase predicha** por el modelo, en ese mismo orden.
* Cada celda `matriz[i][j]` es la cantidad de partidos cuya clase real es la fila `i` y que el modelo predijo como la columna `j`.
* La **diagonal principal** (`matriz[0][0]`, `matriz[1][1]`, `matriz[2][2]`) son los **aciertos**; todo lo que está fuera de la diagonal son **errores de clasificación**.

Con el resultado obtenido:

```
              Predicho: E   Predicho: L   Predicho: V
Real: E       [    5          104            23    ]
Real: L       [    4          161            25    ]
Real: V       [    3           86            62    ]
```

* **Sumando cada fila** se obtiene el total de partidos reales de esa clase (`5+104+23=132` empates, `4+161+25=190` locales, `3+86+62=151` visitantes — coincide con el `support` del `classification_report`). Esa suma, comparada contra el valor de la diagonal, es el **recall** de la clase: por ejemplo, de los 132 empates reales, el modelo solo acertó 5 (`recall E = 5/132 ≈ 0.04`).
* **Sumando cada columna** se obtiene el total de veces que el modelo predijo esa clase. Comparado contra la diagonal, da la **precisión**: de las `5+4+3=12` veces que predijo `E`, acertó 5 (`precisión E = 5/12 ≈ 0.42`).
* **Sumando la diagonal** (`5+161+62=228`) y dividiendo por el total (`473`) se obtiene el **accuracy** global (`228/473 ≈ 0.482`), el mismo valor que reporta `accuracy_score`.

**Lectura para este modelo en particular:**

* La fila de `E` (empates) es la más problemática: de 132 empates reales, **104 fueron clasificados como `L`** y 23 como `V` — el modelo casi nunca predice un empate (columna `E` casi vacía: solo 12 predicciones en total).
* La fila de `V` también sufre bastante: de 151 victorias visitantes, **86 se confunden con `L`**.
* En cambio, la fila de `L` es la más "sana": 161 de 190 locales se predicen correctamente (recall alto), pero a costa de "arrastrar" hacia `L` una gran cantidad de empates y victorias visitantes que en realidad no lo eran (por eso la precisión de `L` es baja, 0.46, pese a su alto recall).

En resumen: el modelo aprendió a **predecir `L` casi por defecto** cuando la evidencia no es muy clara, en vez de distinguir realmente entre las 3 clases — el patrón típico de un clasificador que se apoya demasiado en la clase mayoritaria.


#### ¿Es un buen predictor?

Comparando contra un baseline trivial ("predecir siempre la clase mayoritaria"): en el conjunto de evaluación, `L` es la clase más frecuente con 190 de 473 partidos (≈ 41.7%). Nuestro modelo obtiene **48.2%** de accuracy, es decir, supera a ese baseline por apenas ~6-7 puntos porcentuales.

Sumado a lo observado en la matriz de confusión, la conclusión es que **todavía no es un buen predictor**:

* Es apenas mejor que "apostar siempre al local", y solo un poco mejor que adivinar al azar entre 3 clases (33%).
* Prácticamente **no distingue empates** (recall de `E` = 0.04): con estos 3 atributos (`home`, `away`, `full_time`) el modelo no tiene señal suficiente para detectar cuándo un partido va a terminar igualado, y "resuelve" esa incertidumbre prediciendo `L` por default.
* El ajuste del hiperparámetro `m` (sección de cross-validation) no cambia este panorama: como se explicó, el suavizado casi no influye con un dataset de este tamaño — el problema no es el hiperparámetro, sino que **los atributos actuales no alcanzan** para separar bien las 3 clases (en particular los empates).

Esto es esperable para una primera versión con pocos atributos categóricos crudos, y sirve como **punto de partida** para comparar contra el árbol de decisión y contra `CategoricalNB`/`RandomForestClassifier` de sklearn más adelante.


---
### 7. Comparación con `CategoricalNB` de scikit-learn

`CategoricalNB` no acepta strings ni one-hot: necesita una **codificación ordinal** (enteros 0..k-1 por columna). Además, a diferencia de nuestro `NaiveBayes` propio, no maneja automáticamente valores no vistos en entrenamiento (equipos que solo aparecen en 2024-2025) — si aparece una categoría nueva en `predict`, falla.

Para resolverlo de forma equivalente a nuestro `respaldo_categoricas_`:

* Se usa `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=k)`, reservando un índice extra (`k`) por columna para valores no vistos en entrenamiento.
* Se le informa a `CategoricalNB` esa cantidad real de categorías por columna vía `min_categories`, para que reserve espacio (y probabilidad suavizada) para ese valor "desconocido".
* Se usa `alpha=1.0` (suavizado Laplace estándar de sklearn) como punto de comparación "de fábrica". No es matemáticamente idéntico a nuestro `m` (la relación aproximada es `alpha ≈ m / cantidad_de_categorías`, que varía por columna ya que `home`/`away` tienen muchas más categorías que `full_time`), pero es conceptualmente análogo: ambos suavizan las probabilidades para evitar ceros.


In [34]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

X_train_ord = X_train_nb.copy()
X_test_ord = X_test_nb.copy()
cantidad_categorias = []

for columna in atributos_naive_bayes:
    categorias_train = sorted(X_train_nb[columna].unique())
    valor_desconocido = len(categorias_train)  # índice reservado para valores no vistos en entrenamiento

    encoder = OrdinalEncoder(
        categories=[categorias_train],
        handle_unknown='use_encoded_value',
        unknown_value=valor_desconocido,
    )
    X_train_ord[columna] = encoder.fit_transform(X_train_nb[[columna]])
    X_test_ord[columna] = encoder.transform(X_test_nb[[columna]])
    cantidad_categorias.append(valor_desconocido + 1)

categorical_nb = CategoricalNB(alpha=1.0, min_categories=cantidad_categorias)
categorical_nb.fit(X_train_ord, Y_train)

predicciones_categorical_nb = categorical_nb.predict(X_test_ord)
predicciones_categorical_nb[:5]


array(['L', 'V', 'L', 'V', 'L'], dtype='<U1')

In [35]:
print("Accuracy:", accuracy_score(Y_test, predicciones_categorical_nb))
print()
print(classification_report(Y_test, predicciones_categorical_nb))
print("Matriz de confusión (orden de clases:", list(categorical_nb.classes_), "):")
print(confusion_matrix(Y_test, predicciones_categorical_nb, labels=categorical_nb.classes_))


Accuracy: 0.4820295983086681

              precision    recall  f1-score   support

           E       0.42      0.04      0.07       132
           L       0.46      0.85      0.60       190
           V       0.56      0.41      0.48       151

    accuracy                           0.48       473
   macro avg       0.48      0.43      0.38       473
weighted avg       0.48      0.48      0.41       473

Matriz de confusión (orden de clases: [np.str_('E'), np.str_('L'), np.str_('V')] ):
[[  5 104  23]
 [  4 161  25]
 [  3  86  62]]


In [36]:
from sklearn.metrics import f1_score, recall_score

comparacion = pd.DataFrame([
    {
        'modelo': 'NaiveBayes propio (m=mejor_m)',
        'accuracy': accuracy_score(Y_test, predicciones_nb_optimizado),
        'f1_macro': f1_score(Y_test, predicciones_nb_optimizado, average='macro'),
        'recall_E': recall_score(Y_test, predicciones_nb_optimizado, labels=['E'], average='macro'),
    },
    {
        'modelo': 'CategoricalNB (sklearn, alpha=1.0)',
        'accuracy': accuracy_score(Y_test, predicciones_categorical_nb),
        'f1_macro': f1_score(Y_test, predicciones_categorical_nb, average='macro'),
        'recall_E': recall_score(Y_test, predicciones_categorical_nb, labels=['E'], average='macro'),
    },
])
comparacion


,modelo,accuracy,f1_macro,recall_E
0,NaiveBayes propio (m=mejor_m),0.48203,0.379911,0.037879
1,"CategoricalNB (sklearn, alpha=1.0)",0.48203,0.379911,0.037879


#### Conclusión de la comparación

El `NaiveBayes` propio y `CategoricalNB` de sklearn dan resultados **idénticos** en este dataset: mismo accuracy (0.482), mismo F1-macro (0.380), mismo recall de `E` (0.038) y misma matriz de confusión.

Esto es un buen indicio de que la implementación propia es **matemáticamente correcta**: al usar el mismo criterio de suavizado (Laplace/m-estimate con una constante equivalente) y manejar los valores no vistos de forma análoga (`respaldo_categoricas_` ↔ `min_categories` + `unknown_value`), ambas implementaciones llegan exactamente al mismo resultado.

Como contrapartida, esto también confirma el diagnóstico anterior: el problema **no es la implementación ni el suavizado**, sino que los atributos actuales (`home`, `away`, `full_time`) no le dan a ningún Naive Bayes (propio o de sklearn) la señal necesaria para detectar empates. Mejorar esto requiere agregar atributos nuevos (forma reciente, ELO, etc.), no cambiar de librería.
